In [2]:
import pandas as pd
import csv
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats
import pandas as pd
import urllib.request
import gzip
import ast
from pathlib import Path
from scipy.stats import spearmanr

files = {
    "AML": "/Users/justin.seby/Documents/venv/Justin/Karolinska Institutet/DDLS/DDLS Code/DDLS Projects/LADDER Code Repo/Error analysis/AML Validation V1.csv",
    "Breast Cancer": "/Users/justin.seby/Documents/venv/Justin/Karolinska Institutet/DDLS/DDLS Code/DDLS Projects/LADDER Code Repo/Error analysis/Breast Cancer Validation V1.csv",
    "Lung Cancer": "/Users/justin.seby/Documents/venv/Justin/Karolinska Institutet/DDLS/DDLS Code/DDLS Projects/LADDER Code Repo/Error analysis/Lung Cancer Validation V1.csv",
}

dfs = []
for cancer_type, path in files.items():
    df = pd.read_csv(
        path,
        sep=",",
        quotechar='"',
        engine="python",
    )
    df["Cancer_Type"] = cancer_type
    dfs.append(df)
    print(f"{cancer_type}: {df.shape[0]} rows, {df.shape[1]} columns")
    print(df.columns.tolist())

combined = pd.concat(dfs, ignore_index=True)

print("\nCombined shape:", combined.shape)
print(combined["Cancer_Type"].value_counts())

out_path = "Combined_Validation_V1.csv"
combined.to_csv(out_path, index=False)
print(f"\nSaved to: {out_path}")

combined.head()

AML: 64 rows, 19 columns
['Set_ID', 'Genes', 'Process_With_Enrichment_Original', 'Process_Without_Enrichment_Original', 'Confidence_With_Enrichment_Before', 'Confidence_Without_Enrichment_Before', 'Confidence_With_Enrichment_After', 'Confidence_Without_Enrichment_After', 'Final_Process', 'Final_Confidence', 'Validation_Analysis_Text', 'Supporting_Citations', 'Conflicting_Evidence_Found', 'Conflict_Description', 'Top_Journal_Papers_Used', 'Number_of_Top_Journal_Papers', 'Total_Papers_Found', 'GeneSet_Name', 'Cancer_Type']
Breast Cancer: 139 rows, 19 columns
['Set_ID', 'Genes', 'Process_With_Enrichment_Original', 'Process_Without_Enrichment_Original', 'Confidence_With_Enrichment_Before', 'Confidence_Without_Enrichment_Before', 'Confidence_With_Enrichment_After', 'Confidence_Without_Enrichment_After', 'Final_Process', 'Final_Confidence', 'Validation_Analysis_Text', 'Supporting_Citations', 'Conflicting_Evidence_Found', 'Conflict_Description', 'Top_Journal_Papers_Used', 'Number_of_Top_Journ

,Set_ID,Genes,Process_With_Enrichment_Original,Process_Without_Enrichment_Original,Confidence_With_Enrichment_Before,Confidence_Without_Enrichment_Before,Confidence_With_Enrichment_After,Confidence_Without_Enrichment_After,Final_Process,Final_Confidence,Validation_Analysis_Text,Supporting_Citations,Conflicting_Evidence_Found,Conflict_Description,Top_Journal_Papers_Used,Number_of_Top_Journal_Papers,Total_Papers_Found,GeneSet_Name,Cancer_Type
0,4,"['ABCB9', 'ABCC2', 'ACAA2', 'ACOX1', 'ACOX3', ...",SRP-Dependent Cotranslational Protein Targetin...,Transcriptional Regulation and Chromatin Remod...,0.85,0.75,0.05,0.85,Transcriptional Regulation and Chromatin Remod...,0.85,The validation process strictly adhered to the...,"['""Gene expression profiling of acute myeloid ...",False,No conflicts detected,Paper 1: 'Cross-cancer profiling of molecular ...,50,50,YAGI_AML_WITH_INV_16_TRANSLOCATION,AML
1,7,"['ABCC1', 'ABR', 'ACADSB', 'ACKR3', 'ACOT13', ...",*Dysregulated Immune and Cytokine Signaling in...,*Oncogenic Transcription and Epigenetic Dysreg...,0.85,0.78,0.90,0.85,Dysregulated Immune and Cytokine Signaling in ...,0.90,"The validation process involved a rigorous, li...","['""Single-cell analysis reveals altered tumor ...",False,No conflicts detected,Paper 1: 'Single-cell analysis reveals altered...,50,50,YAGI_AML_WITH_T_8_21_TRANSLOCATION,AML
2,9,"['ABCC6', 'ABHD3', 'ABR', 'ACTA1', 'ADA', 'ADA...",B Cell Receptor Signaling and B Cell Activatio...,Integrated Signaling Hub Involving Kinase/Phos...,0.85,0.70,0.10,0.85,Integrated Signaling Hub Involving Kinase/Phos...,0.85,The validation process strictly adhered to the...,"['""Single-cell analysis of immune recognition ...",False,No conflicts detected,Paper 1: 'Novel Diagnostic and Therapeutic Opt...,50,50,YAGI_AML_WITH_11Q23_REARRANGED,AML
3,15,"['ABCB1', 'ABLIM1', 'ACSL4', 'ADCY3', 'ADIPOR1...",Hematopoietic Differentiation and Myeloid Line...,Transcriptional Dysregulation and Signaling in...,0.85,0.75,0.90,0.85,Hematopoietic Differentiation and Myeloid Line...,0.90,"The validation process involved a rigorous, li...",[],False,No conflicts detected,Paper 1: 'Single-cell analysis reveals altered...,50,50,VERHAAK_AML_WITH_NPM1_MUTATED_DN,AML
4,22,"['ACO1', 'ADAMTS3', 'AHCYL1', 'AHNAK', 'AIF1',...","Myeloid Cell Adhesion, Migration, and Immune R...",Hematopoietic Stem/Progenitor Cell Regulation ...,0.85,0.80,0.25,0.85,Hematopoietic Stem/Progenitor Cell Regulation ...,0.85,"The validation process involved a strict, lite...",[],False,No conflicts detected,Paper 1: 'Genomic subtyping and therapeutic ta...,50,50,YAGI_AML_FAB_MARKERS,AML


In [3]:
data_dir = Path("./ncbi_generif_data")
data_dir.mkdir(exist_ok=True)

generif_url = "https://ftp.ncbi.nih.gov/gene/GeneRIF/generifs_basic.gz"
gene_info_url = "https://ftp.ncbi.nih.gov/gene/DATA/GENE_INFO/Mammalia/Homo_sapiens.gene_info.gz"

generif_path = data_dir / "generifs_basic.gz"
gene_info_path = data_dir / "Homo_sapiens.gene_info.gz"

if not generif_path.exists():
    print("Downloading generifs_basic.gz ...")
    urllib.request.urlretrieve(generif_url, generif_path)

if not gene_info_path.exists():
    print("Downloading Homo_sapiens.gene_info.gz ...")
    urllib.request.urlretrieve(gene_info_url, gene_info_path)

print("Downloads done.")

gene_info_cols = [
    "tax_id", "GeneID", "Symbol", "LocusTag", "Synonyms", "dbXrefs",
    "chromosome", "map_location", "description", "type_of_gene",
    "Symbol_from_nomenclature_authority", "Full_name_from_nomenclature_authority",
    "Nomenclature_status", "Other_designations", "Modification_date", "Feature_type"
]

gene_info = pd.read_csv(
    gene_info_path, sep="\t", names=gene_info_cols, header=0,
    dtype=str, na_values=["-"]
)
gene_info = gene_info[gene_info["tax_id"] == "9606"]

symbol_to_geneid = dict(zip(gene_info["Symbol"], gene_info["GeneID"]))

synonym_to_geneid = {}
for _, row in gene_info.iterrows():
    if pd.notna(row["Synonyms"]):
        for syn in row["Synonyms"].split("|"):
            synonym_to_geneid.setdefault(syn, row["GeneID"])

print(f"Loaded {len(symbol_to_geneid)} human gene symbols.")

generif_cols = ["tax_id", "GeneID", "PubMed_ID", "timestamp", "GeneRIF_text"]

generifs = pd.read_csv(
    generif_path, sep="\t", names=generif_cols, header=0,
    dtype=str
)
generifs = generifs[generifs["tax_id"] == "9606"]

generif_counts = generifs.groupby("GeneID").size().to_dict()
pubmed_counts = generifs.groupby("GeneID")["PubMed_ID"].nunique().to_dict()

print(f"GeneRIF counts available for {len(generif_counts)} human genes.")
def gene_to_generif_count(symbol):
    gene_id = symbol_to_geneid.get(symbol) or synonym_to_geneid.get(symbol)
    if gene_id is None:
        return None
    return generif_counts.get(gene_id, 0)

def gene_to_pubmed_count(symbol):
    gene_id = symbol_to_geneid.get(symbol) or synonym_to_geneid.get(symbol)
    if gene_id is None:
        return None
    return pubmed_counts.get(gene_id, 0)

combined = pd.read_csv("Combined_Validation_V1.csv")

def parse_gene_list(genes_str):
    try:
        return ast.literal_eval(genes_str)
    except (ValueError, SyntaxError):
        return []

combined["Gene_List"] = combined["Genes"].apply(parse_gene_list)

def score_gene_set(gene_list):
    counts = [gene_to_generif_count(g) for g in gene_list]
    found_counts = [c for c in counts if c is not None]
    n_not_found = sum(1 for c in counts if c is None)
    if not found_counts:
        return pd.Series({
            "GeneRIF_total": 0, "GeneRIF_mean": 0, "GeneRIF_median": 0,
            "Genes_not_found": n_not_found, "Genes_found": 0
        })
    return pd.Series({
        "GeneRIF_total": sum(found_counts),
        "GeneRIF_mean": sum(found_counts) / len(found_counts),
        "GeneRIF_median": pd.Series(found_counts).median(),
        "Genes_not_found": n_not_found,
        "Genes_found": len(found_counts),
    })

generif_scores = combined["Gene_List"].apply(score_gene_set)
combined = pd.concat([combined, generif_scores], axis=1)

combined.to_csv("Combined_Validation_V1_with_GeneRIF.csv", index=False)
combined[["Set_ID", "GeneSet_Name", "Final_Confidence", "GeneRIF_total", "GeneRIF_mean", "GeneRIF_median", "Genes_found", "Genes_not_found"]].head(10)

Downloads done.
Loaded 193699 human gene symbols.
GeneRIF counts available for 20759 human genes.


,Set_ID,GeneSet_Name,Final_Confidence,GeneRIF_total,GeneRIF_mean,GeneRIF_median,Genes_found,Genes_not_found
0,4,YAGI_AML_WITH_INV_16_TRANSLOCATION,0.85,39690.0,96.804878,22.5,410.0,1.0
1,7,YAGI_AML_WITH_T_8_21_TRANSLOCATION,0.90,49515.0,133.104839,32.0,372.0,1.0
2,9,YAGI_AML_WITH_11Q23_REARRANGED,0.85,29391.0,85.688047,24.0,343.0,0.0
3,15,VERHAAK_AML_WITH_NPM1_MUTATED_DN,0.90,42735.0,170.940000,41.0,250.0,0.0
4,22,YAGI_AML_FAB_MARKERS,0.85,18671.0,96.740933,38.0,193.0,0.0
5,23,VERHAAK_AML_WITH_NPM1_MUTATED_UP,0.90,46766.0,250.085561,59.0,187.0,0.0
6,24,ALCALAY_AML_BY_NPM1_LOCALIZATION_DN,0.95,27933.0,150.177419,37.0,186.0,0.0
7,29,FIGUEROA_AML_METHYLATION_CLUSTER_3_UP,0.85,8682.0,51.070588,9.0,170.0,0.0
8,35,ALCALAY_AML_BY_NPM1_LOCALIZATION_UP,0.85,16719.0,119.421429,24.5,140.0,0.0
9,36,FIGUEROA_AML_METHYLATION_CLUSTER_6_UP,0.85,5811.0,41.507143,9.0,140.0,0.0


In [8]:
combined = pd.read_csv("Combined_Validation_V1_with_GeneRIF.csv")

metrics = [
    "GeneRIF_total",
    "GeneRIF_mean",
    "GeneRIF_median"
]

for metric in metrics:
    rho, p = spearmanr(combined[metric], combined["Final_Confidence"])
    print(f"{metric}: rho={rho:.5f}, p={p:.12f}")

GeneRIF_total: rho=0.50108, p=0.000000000000
GeneRIF_mean: rho=0.32095, p=0.000000354346
GeneRIF_median: rho=0.35168, p=0.000000020074


In [5]:
for cancer, df in combined.groupby("Cancer_Type"):
    rho, p = spearmanr(df["GeneRIF_mean"], df["Final_Confidence"])
    print(f"{cancer}: rho={rho:.3f}, p={p:.4f}")

AML: rho=0.125, p=0.3241
Breast Cancer: rho=0.359, p=0.0000
Lung Cancer: rho=0.458, p=0.0038
